
# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the geolocation dataset as part of the Data Quality Monitoring System for e-commerce operations. It performs data inspection, profiling, cleaning, and validation to identify and address data quality issues before the dataset is used for downstream analysis.

This script focuses on the preparation of the Olist order payments dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports payment analysis by capturing the payment methods, installment information, and transaction values associated with each customer order, providing a reliable foundation for analyzing customer payment behavior, payment trends, and revenue performance in SQL and Power BI.

### Import The Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load The Dataset

In [2]:
order_payments_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_payments_dataset.csv")

## **ORDER PAYMENTS DATASET**

### 1. Data Inspection

In [ ]:
# The first five rows of the order payments dataset
order_payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
# The number of rows and columns in the order payments dataset
order_payments_df.shape

(103886, 5)

In [ ]:
# The column names and data types in the order payments dataset
order_payments_df.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

The data type assessment indicates that all variables within the order payments dataset are stored using appropriate data types that accurately represent their intended business purpose. The **`order_id`** and **`payment_type`** columns are stored as text (`object`), which is suitable for categorical and identifier variables. The **`payment_sequential`** and **`payment_installments`** columns are stored as integers (`int64`), reflecting whole-number payment sequences and installment counts, while **`payment_value`** is stored as a floating-point number (`float64`) to accommodate monetary values containing decimal places. Based on this assessment, no data type conversions are required during the data preparation process.

In [ ]:
# The number of missing values in each column of the order payments dataset
order_payments_df.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

The missing value assessment indicates that the order payments dataset contains **no missing values** across any of its variables. All payment records contain complete information for the **`order_id`**, **`payment_sequential`**, **`payment_type`**, **`payment_installments`**, and **`payment_value`** columns. The absence of missing values demonstrates a high level of data completeness and suggests that no imputation, record removal, or other missing value treatment is required during the data preparation process.

In [11]:
# The number of duplicate rows in the order payments dataset
for col in order_payments_df.columns:
    duplicates = order_payments_df[col].duplicated().sum()
    print(f"{col} : {duplicates}")

order_id : 4446
payment_sequential : 103857
payment_type : 103881
payment_installments : 103862
payment_value : 74809


The duplicate value assessment identified repeated values across several individual columns within the order payments dataset. Specifically, the **`order_id`** column contains **4,446** duplicate values, while the **`payment_sequential`**, **`payment_type`**, **`payment_installments`**, and **`payment_value`** columns also contain repeated values. At this stage, the presence of duplicate values within individual columns should not be interpreted as duplicate records or data quality issues, as repeated values may naturally occur within transactional payment data. Further investigation during the data profiling phase will determine whether these repeated values represent legitimate business scenarios or require additional data quality treatment. Therefore, no cleaning action is recommended based solely on the duplicate value assessment.

In [ ]:
# The summary statistics of the order payments dataset numeric columns
order_payments_df.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


Descriptive statistics indicate that the order payments dataset contains complete payment information with substantial variation in payment amounts and installment usage. The prevalence of single-payment transactions and credit card payments aligns with expected e-commerce purchasing behavior. Potential anomalies, including zero-value payments and unusually large transactions, should be investigated during the data profiling phase to determine whether they represent legitimate business events or data quality issues.

In [ ]:
# The summary statistics of the order payments dataset object columns
order_payments_df.describe(include='object')

,order_id,payment_type
count,103886,103886
unique,99440,5
top,fa65dad1b0e818e3ccc5cb0e39231352,credit_card
freq,29,76795


The summary statistics for the categorical variables indicate that the order payments dataset contains **103,886** payment records associated with **99,440** unique **`order_id`** values and **five** distinct payment methods. The most frequently occurring payment method is **`credit_card`**, accounting for **76,795** payment transactions, while the most frequently occurring **`order_id`** appears **29** times within the dataset. At this stage, these descriptive statistics provide an overview of the dataset's categorical characteristics and should not be interpreted as data quality issues. The frequency distributions and repeated **`order_id`** values will be examined further during the data profiling phase to determine whether they represent expected business processes or require additional investigation.

### 2. Data Profiling

#### Duplicated Values

In [20]:
# Display duplicate records
duplicate_records = order_payments_df[
    order_payments_df.duplicated(keep=False)
].sort_values(
    by=["order_id", "payment_sequential"]
)

duplicate_records

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [21]:
# Display the number of duplicate records
duplicate_records.shape

(0, 5)

The duplicate record assessment confirms that the order payments dataset contains no duplicate records. The duplicate count returned zero, and the duplicate record query produced an empty DataFrame, indicating that each payment transaction is unique across all variables. Although duplicate values were identified within individual columns during the data inspection phase, these repetitions are expected in transactional payment data and do not constitute duplicate observations. Therefore, no duplicate records require removal during the data preparation process.

#### Order ID Duplicates

In [22]:
# Count the number of payment records associated with each order
order_payment_counts = (
    order_payments_df.groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

order_payment_counts.describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [23]:
# Display orders associated with more than one payment record
order_payment_counts[order_payment_counts > 1]

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
                                    ..
fb662b5ba362e72d75d486ca4d29c9e4     2
a46e7e8e02915adecb392ad065121a51     2
6d5e1e480d92f1bde5e9eb187fa388d5     2
a61379db8a77b3c23fd2f0f60a5bd564     2
2745579b0446d51b9aa8263485dbeb1d     2
Length: 2961, dtype: int64

In [25]:
# Display the number of orders with multiple payment records
print((order_payment_counts > 1).sum())

2961


In [26]:
# Inspect the payment records for the order with the highest number of payment transactions
order_payments_df[
    order_payments_df["order_id"] == order_payment_counts.idxmax()
].sort_values("payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


The order with the highest number of payment records has **29 payment transactions**, all using vouchers. There are also zero-value voucher payments mixed in with positive values. That's unusual.

In [27]:
# Calculate the total payment value for the order with the highest number of payment transactions
order_payments_df[
    order_payments_df["order_id"] == order_payment_counts.idxmax()
]["payment_value"].sum()

np.float64(457.99)

In [28]:
# Display payment records with a running total
highest_order = order_payments_df[
    order_payments_df["order_id"] == order_payment_counts.idxmax()
].sort_values("payment_sequential")

highest_order = highest_order.assign(
    cumulative_payment=highest_order["payment_value"].cumsum()
)

highest_order

,order_id,payment_sequential,payment_type,payment_installments,payment_value,cumulative_payment
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51,12.22
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95,15.17
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16,44.33
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66,44.99
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02,50.01
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32,50.33
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02,76.35
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08,77.43
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86,90.29


- The dataset contains **99,440 unique orders** and **103,886 payment records**, indicating that some orders are associated with more than one payment transaction.
- A total of **2,961 orders** have multiple payment records.
The highest number of payment transactions associated with a single order is 29.
- Inspection of the order with **29 payment records** shows that each payment has a unique `payment_sequential` value, increasing sequentially from 1 onwards.
- The cumulative payment values increase logically throughout the transaction history, resulting in a total payment amount of **457.99**.
- Although two payment transactions have a payment value of **0.00**, there is insufficient evidence at this stage to conclude that these records represent data quality issues. They will be investigated further during the Payment Value profiling section.

The Order ID investigation indicates that repeated **`order_id`** values represent legitimate business transactions rather than duplicate observations. While 2,961 orders are associated with multiple payment records, inspection of the order containing the highest number of payment transactions demonstrates a logical payment sequence and a cumulative payment value of **457.99**. These findings suggest that a single order may be settled through multiple payment transactions, which is consistent with expected payment processing behavior. Although two payment records with a value of **0.00** were observed within the inspected order, their validity cannot be determined at this stage and will be investigated separately during the payment value assessment. Therefore, no cleaning action is required for repeated **`order_id`** values.

#### Payment Sequence Consistency

In [29]:
# Summary statistics for payment sequence
order_payments_df["payment_sequential"].describe()

count    103886.000000
mean          1.092679
std           0.706584
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          29.000000
Name: payment_sequential, dtype: float64

In [30]:
# Distribution of payment sequence values
order_payments_df["payment_sequential"].value_counts().sort_index()

payment_sequential
1     99360
2      3039
3       581
4       278
5       170
6       118
7        82
8        54
9        43
10       34
11       29
12       21
13       13
14       10
15        8
16        6
17        6
18        6
19        6
20        4
21        4
22        3
23        2
24        2
25        2
26        2
27        1
28        1
29        1
Name: count, dtype: int64

In [31]:
# Compare the number of payment records with the maximum payment sequence for each order
payment_sequence_check = (
    order_payments_df.groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count"),
        max_sequence=("payment_sequential", "max")
    )
)

payment_sequence_check

,payment_records,max_sequence
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,1,1
00018f77f2f0320c557190d7a144bdd3,1,1
000229ec398224ef6ca0657da4fc703e,1,1
00024acbcdf0a6daa1e931b038114c75,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1
...,...,...
fffc94f6ce00a00581880bf54a75a037,1,1
fffcd46ef2263f404302a634eb57f7eb,1,1
fffce4705a9662cd70adb13d4a31832d,1,1


In [32]:
# Identify orders where the number of payment records does not match the maximum payment sequence
payment_sequence_check[
    payment_sequence_check["payment_records"] != payment_sequence_check["max_sequence"]
]

,payment_records,max_sequence
order_id,,
00ac05fe0fc047c54418098eb64e3aaa,1,2
056c68d093c100017aab1f00f260705c,1,2
0668d086b3ae41adde3aed3dacbc8fae,1,2
0a7d898c6305101e69e9f5a05cd0130d,1,2
0d0e88a418636d40ea780339701db49c,1,2
...,...,...
f0f94b7c7548150f33f5d9e7597d396f,1,2
f17a58e669a7f4cf7a7f227899f8abf7,1,2
f5691c2b1ca263490374d13d020bd950,1,2


**80 orders have one payment record but a maximum payment sequence of 2.**

In [33]:
# Display payment records for orders with inconsistent payment sequences
inconsistent_orders = payment_sequence_check[
    payment_sequence_check["payment_records"] != payment_sequence_check["max_sequence"]
].index

order_payments_df[
    order_payments_df["order_id"].isin(inconsistent_orders)
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
27191,00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
12785,056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
40318,0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74
84637,0a7d898c6305101e69e9f5a05cd0130d,2,credit_card,5,215.47
82233,0d0e88a418636d40ea780339701db49c,2,debit_card,1,121.09
...,...,...,...,...,...
70969,f0f94b7c7548150f33f5d9e7597d396f,2,debit_card,1,148.95
49746,f17a58e669a7f4cf7a7f227899f8abf7,2,debit_card,1,230.15
6200,f5691c2b1ca263490374d13d020bd950,2,debit_card,1,172.98
15767,fce3db7092e1f432d0ec790414b6a5e4,2,debit_card,1,66.38


In [34]:
# Summarise inconsistent payment sequence combinations
payment_sequence_check[
    payment_sequence_check["payment_records"] != payment_sequence_check["max_sequence"]
].value_counts()

payment_records  max_sequence
1                2               78
2                3                2
Name: count, dtype: int64

- 78 orders have:
    - `payment_records = 1`
    - `max_sequence = 2`
- 2 orders have:
    - `payment_records = 2`
    - `max_sequence = 3`

And when we inspected the original records, the pattern has became even clearer:

- The payment sequences **start at 2** instead of 1.
- There are **no corresponding sequence 1 records** in the payments dataset for those orders. 

In [35]:
# Distribution of payment methods for inconsistent payment sequences
order_payments_df[
    order_payments_df["order_id"].isin(inconsistent_orders)
]["payment_type"].value_counts()

payment_type
debit_card     52
credit_card    29
boleto          1
Name: count, dtype: int64

In [36]:
# Summary statistics for inconsistent payment sequences
order_payments_df[
    order_payments_df["order_id"].isin(inconsistent_orders)
].groupby("payment_type").agg(
    payment_records=("order_id", "count"),
    average_payment=("payment_value", "mean"),
    min_payment=("payment_value", "min"),
    max_payment=("payment_value", "max")
)

,payment_records,average_payment,min_payment,max_payment
payment_type,,,,
boleto,1,94.400000,94.40,94.40
credit_card,29,215.127241,37.58,828.65
debit_card,52,136.746154,20.00,847.38


The payment sequence assessment indicates that most payment transactions follow a logical sequence, with the majority of orders containing a single payment transaction assigned a payment sequence of **1**.

However, a small number of orders exhibit inconsistencies between the number of payment records and the recorded payment sequence values. Specifically, 80 orders contain payment sequences that do not begin at the expected starting point, with most affected records beginning at sequence**2** despite only one recorded payment transaction. 

These inconsistencies occur across multiple payment methods, suggesting that they are not associated with a single payment type. As there is insufficient evidence to determine whether these observations represent data quality issues or characteristics of the original payment processing system, no cleaning action is recommended. The original payment sequence values should therefore be retained and documented as part of the dataset's known characteristics.

#### Payment Value 

In [37]:
# Display payment records with zero payment values
zero_payment_values = order_payments_df[
    order_payments_df["payment_value"] == 0
]

zero_payment_values

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [38]:
# Display the number of zero payment transactions
zero_payment_values.shape

(9, 5)

In [39]:
# Summarise payment methods associated with zero payment transactions
zero_payment_values["payment_type"].value_counts()

payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

In [40]:
# Display descriptive statistics for payment values
order_payments_df["payment_value"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
50%         100.000000
90%         297.270000
95%         437.635000
99%        1039.916500
max       13664.080000
Name: payment_value, dtype: float64

- A total of **9** payment transactions have a payment value of 0.00.
- These zero-value transactions are associated with only two payment methods:
    - **voucher (6)**
    - **not_defined (3)**

In [41]:
# Display all payment records for orders containing zero-value payments
zero_payment_orders = zero_payment_values["order_id"].unique()

order_payments_df[
    order_payments_df["order_id"].isin(zero_payment_orders)
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


In [42]:
# Calculate the total payment value for orders containing zero-value payments
order_payments_df[
    order_payments_df["order_id"].isin(zero_payment_orders)
].groupby("order_id")["payment_value"].sum()

order_id
00b1cb0320190ca0daa2c88b35206009      0.00
45ed6e85398a87c253db47c2d9f48216     71.14
4637ca194b6387e2d538dc89b124b0ee      0.00
6ccb433e00daae1283ccc956189c82ae    122.04
8bcbe01d44d147f901cd3192671144db     74.16
b23878b3e8eb4d25a158f57d96331b18    171.57
c8c528189310eaa44a745b8d9d26908b      0.00
fa65dad1b0e818e3ccc5cb0e39231352    457.99
Name: payment_value, dtype: float64

The payment value assessment identified nine payment transactions with a recorded payment value of **0.00**. Further investigation revealed that six of these transactions occur alongside additional positive payment transactions within the same order, indicating that they form part of legitimate multi-payment transactions rather than representing missing monetary values. The remaining three transactions are standalone records with a payment type of **`not_defined`**, resulting in an overall order payment value of **0.00**. As the available data does not provide sufficient evidence to determine whether these records represent data quality issues or valid business events captured by the source system, no cleaning action is recommended. 

#### Payment Installments Patterns

In [43]:
# Display the distribution of payment installment values
order_payments_df["payment_installments"].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [44]:
# Display payment records with zero installments
zero_installments = order_payments_df[
    order_payments_df["payment_installments"] == 0
]

zero_installments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [45]:
# Display the number of zero-installment payment records
zero_installments.shape

(2, 5)

In [46]:
# Summarise payment methods associated with zero installments
zero_installments["payment_type"].value_counts()

payment_type
credit_card    2
Name: count, dtype: int64

- **2 records** with payment_installments = 0
- Both use **credit_card**
- Both have **payment_sequential = 2**
- Both have positive payment values

In [47]:
# Display all payment records for orders containing zero installments
zero_installment_orders = zero_installments["order_id"].unique()

order_payments_df[
    order_payments_df["order_id"].isin(zero_installment_orders)
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [48]:
# Compare installment values within the affected orders
order_payments_df[
    order_payments_df["order_id"].isin(zero_installment_orders)
].groupby("order_id").agg(
    payment_records=("order_id", "count"),
    payment_sequences=("payment_sequential", list),
    installment_values=("payment_installments", list),
    payment_types=("payment_type", list)
)

,payment_records,payment_sequences,installment_values,payment_types
order_id,,,,
1a57108394169c0b47d8f876acc9ba2d,1,[2],[0],[credit_card]
744bade1fcf9ff3f31d860ace076d422,1,[2],[0],[credit_card]


- The installment assessment identified two payment records with a recorded installment value of 0.
- Both records are associated with the credit_card payment method.
- Further investigation shows that each affected order contains a single payment record with **payment_sequential = 2** and **payment_installments = 0**.
- These records correspond to the payment sequence anomalies identified during the previous profiling step, suggesting that the zero-installment values are associated with the same underlying characteristic rather than representing an independent data quality issue.
- As there is insufficient evidence to determine the intended installment values, the original records should be preserved.

#### Payment Type distribution

In [49]:
# Display the distribution of payment types
order_payments_df["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [51]:
# Display payment records with the 'not_defined' payment type
not_defined_payments = order_payments_df[
    order_payments_df["payment_type"] == "not_defined"
]

not_defined_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [52]:
# Display the number of 'not_defined' payment records
not_defined_payments.shape

(3, 5)

In [53]:
# Summarise the characteristics of 'not_defined' payment records
not_defined_payments[
    ["payment_sequential", "payment_installments", "payment_value"]
].describe()

,payment_sequential,payment_installments,payment_value
count,3.0,3.0,3.0
mean,1.0,1.0,0.0
std,0.0,0.0,0.0
min,1.0,1.0,0.0
25%,1.0,1.0,0.0
50%,1.0,1.0,0.0
75%,1.0,1.0,0.0
max,1.0,1.0,0.0


- The not_defined payment type appears in only 3 records out of 103,886 payment transactions.
- All three not_defined records share identical characteristics:
    - **payment_sequential = 1**
    - **payment_installments = 1**
    - **payment_value = 0.00**
- The consistency across these records suggests that not_defined is likely an original category from the source system rather than a result of inconsistent data entry.
- Although the payment value is zero for all three records, there is insufficient evidence to conclude that the not_defined payment type represents invalid or erroneous data.

### 3. Data Cleaning

The duplicate record assessment confirmed that the order payments dataset contains no duplicate records. Although duplicate values were observed within individual columns, every payment transaction is unique across all variables. Consequently, no duplicate records require removal, and all records will be retained in the cleaned dataset.

The payment sequence assessment identified a small number of records with payment sequence values that do not begin at the expected starting point. As there is insufficient evidence to determine whether these observations represent data quality issues or characteristics of the original payment processing system, no modifications will be made to the payment sequence values. The original values will be retained and documented as part of the dataset's known characteristics.

The payment installment assessment identified two records with an installment value of **0**. These records correspond to the payment sequence anomalies identified during profiling, and no evidence was found to justify modifying the original installment values. Therefore, all installment values will be retained in the cleaned dataset.

The payment type assessment identified three records with a payment type of **`not_defined`**. The consistent characteristics of these records suggest that this value originates from the source system rather than from inconsistent data entry. Consequently, the original payment type values will be preserved without modification.

In [54]:
from IPython.display import display

# Display column names
display(order_payments_df.columns)

# Display data types
display(order_payments_df.dtypes)

# Display missing values
display(order_payments_df.isnull().sum())

# Display duplicate records
display(order_payments_df.duplicated().sum())

# Validate payment sequence anomalies
payment_sequence_check = (
    order_payments_df.groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count"),
        max_sequence=("payment_sequential", "max")
    )
)

payment_sequence_anomalies = (
    payment_sequence_check["payment_records"]
    != payment_sequence_check["max_sequence"]
).sum()

# Validate zero payment values
zero_payment_values = (
    order_payments_df["payment_value"] == 0
).sum()

# Validate zero installment values
zero_installments = (
    order_payments_df["payment_installments"] == 0
).sum()

# Validate 'not_defined' payment types
not_defined_payments = (
    order_payments_df["payment_type"] == "not_defined"
).sum()

print(f"Payment Sequence Anomalies: {payment_sequence_anomalies}")
print(f"Zero Payment Values: {zero_payment_values}")
print(f"Zero Installment Values: {zero_installments}")
print(f"'not_defined' Payment Types: {not_defined_payments}")

# Display DataFrame information
order_payments_df.info()

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

np.int64(0)

Payment Sequence Anomalies: 80
Zero Payment Values: 9
Zero Installment Values: 2
'not_defined' Payment Types: 3
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


### 4. Export The Cleaned Dataset

In [55]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [56]:
order_payments_df.to_csv(
    CLEANED_DATA_PATH / "olist_order_payments_cleaned.csv",
    index=False
)